In [1]:
using LinearAlgebra
using QuantumToolbox
using GLMakie


In [9]:
H_gate = Qobj([1/sqrt(2) 1/sqrt(2); 1/sqrt(2) -1/sqrt(2)], type = Operator())
S_gate = Qobj([1.0+0.0im 0.0+0.0im; 0.0+0.0im 0.0+1.0im], type = Operator())
X_gate = Qobj([0.0+0.0im 1.0+0.0im; 1.0+0.0im 0.0+0.0im], type = Operator())
Z_gate = Qobj([1.0+0.0im 0.0+0.0im; 0.0+0.0im -1.0+0.0im], type = Operator())

function generate_coordinates(initial, controls_array; steps = 40, pause = 5)
    function getxyz(state)
        return [
            Real(expect(sigmax(), state)), 
            Real(expect(sigmay(), state)), 
            Real(expect(sigmaz(), state))
        ]
    end

    timeline = Vector{Vector{Float64}}()
    current = initial
    push!(timeline, getxyz(current))
    
    for c in controls_array
        H_matrix = im * log(Matrix(c.data))
        for i in 1:steps
            fraction = i / steps
            U_fraction = exp(im * H_matrix * fraction)
            interp_state = Qobj(U_fraction * current.data, type = Ket())
            push!(timeline, getxyz(interp_state))
        end
        current = Qobj(exp(im * H_matrix) * current.data, type = Ket())
        push!(timeline, getxyz(current))
        for i in 1:pause
            push!(timeline, getxyz(current))
        end
    end
    return timeline
end

generate_coordinates (generic function with 1 method)

In [ ]:
function animate(initial, controls_array)
    coords_timeline = generate_coordinates(initial, controls_array)
    b = Bloch()
    fig, lscene = render(b)
    
    first_frame = coords_timeline[1]
    line_points = Observable([Point3f(0, 0, 0), Point3f(first_frame[1], first_frame[2], first_frame[3])])
    lines!(lscene, line_points, color = :pink, linewidth = 10)

    record(fig, "quantumC2.mp4", coords_timeline; framerate = 30) do frame_coords
        line_points[] = [Point3f(0, 0, 0), Point3f(frame_coords[1], frame_coords[2], frame_coords[3])]
    end
    println("Done")
end

# start_state = basis(2, 0) 
# controls = [H_gate, S_gate]
# animate(start_state, controls)


animate (generic function with 1 method)

In [55]:
function animate(initial, controls_array)
    coords_timeline = generate_coordinates(initial, controls_array)
    b = Bloch()
    fig, lscene = render(b)
    
    first_frame = coords_timeline[1]
    line_points = Observable(Point3f(first_frame[1], first_frame[2], first_frame[3]))
    println(line_points)
    lines!(lscene, line_points, color = :pink, linewidth = 10)

    record(fig, "quantumC3.mp4", coords_timeline; framerate = 30) do frame_coords
        line_points[] = Point3f(frame_coords[1], frame_coords[2], frame_coords[3])
    end
    println("Done")
end


animate (generic function with 1 method)

In [56]:
start_state = basis(2, 0) 
controls = [H_gate, Z_gate]
animate(start_state, controls)

Observable(Float32[0.0, 0.0, 1.0])
Done
